# RolloTree: Advanced Usage

This notebook covers advanced features beyond the basics:

1. **Depth-level analysis** — `depth_results_` for training dynamics
2. **Preprocessing raw data** — `make_data_binary()` from scratch
3. **Solver tuning** — MIP gap, time limits, Big-M penalty
4. **Parallel execution** — `n_jobs` for faster deep trees
5. **Tree internals** — branch nodes, leaf distributions, pruned paths
6. **Numba acceleration** — JIT-compiled routing performance
7. **Custom analysis** — per-leaf statistics, confusion by leaf, depth vs accuracy curves
8. **Comparing impurity criteria** — Gini vs misclassification across depths

In [1]:
import time
import numpy as np
import pandas as pd
from rollotree import (
    RollingOCT,
    SolverConfig,
    SolverStatus,
    DecisionTree,
    GiniCriterion,
    MisclassificationCriterion,
    export_text,
    export_graphviz,
)
from rollotree.tree._numba import HAS_NUMBA

In [2]:
# Load the bundled Wine dataset
train = pd.read_csv("../rollotree/data/train.csv")
test = pd.read_csv("../rollotree/data/test.csv")

X_train = train.drop("y", axis=1)
y_train = train["y"]
X_test = test.drop("y", axis=1)
y_test = test["y"]

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Classes: {sorted(y_train.unique())} (counts: {dict(y_train.value_counts().sort_index())})")

Train: 160 samples, 130 features
Test:  18 samples
Classes: [1, 2, 3] (counts: {1: 53, 2: 64, 3: 43})


## 1. Depth-Level Analysis with `depth_results_`

The rolling algorithm builds trees level by level. After fitting, `depth_results_` gives you accuracy and timing at each depth — useful for understanding when the tree stops improving.

In [3]:
# Train a depth-5 tree and examine per-depth metrics
model = RollingOCT(depth=5, solver="highs")
model.fit(X_train, y_train)

print(f"{'Depth':>5} {'Train Acc':>10} {'Time (s)':>10} {'Leaves':>8}")
print("-" * 38)

for d in sorted(model.depth_results_):
    r = model.depth_results_[d]
    # We can't get leaves at intermediate depths easily, but show what we have
    print(f"{r.depth:5d} {r.training_accuracy:10.4f} {r.elapsed_time:10.3f}")

print(f"\nFinal test accuracy: {model.score(X_test, y_test):.4f}")
print(f"Final tree: {model.get_n_leaves()} leaves, depth {model.get_depth()}")

Depth  Train Acc   Time (s)   Leaves
--------------------------------------
    2     0.5875      1.653
    3     0.6937      1.617
    4     0.7500      1.549
    5     0.8375      2.117

Final test accuracy: 0.8889
Final tree: 4 leaves, depth 5


In [4]:
# Depth vs accuracy curve — compare train accuracy at each depth
depths = sorted(model.depth_results_.keys())
train_accs = [model.depth_results_[d].training_accuracy for d in depths]
times = [model.depth_results_[d].elapsed_time for d in depths]

print("Depth vs Training Accuracy:")
for d, acc, t in zip(depths, train_accs, times):
    bar = "█" * int(acc * 40)
    print(f"  d={d}: {bar} {acc:.3f}  ({t:.2f}s)")

print(f"\nAccuracy gain from d=2 to d={depths[-1]}: "
      f"+{train_accs[-1] - train_accs[0]:.3f}")

Depth vs Training Accuracy:
  d=2: ███████████████████████ 0.588  (1.65s)
  d=3: ███████████████████████████ 0.694  (1.62s)
  d=4: ██████████████████████████████ 0.750  (1.55s)
  d=5: █████████████████████████████████ 0.838  (2.12s)

Accuracy gain from d=2 to d=5: +0.250


## 2. Preprocessing Raw Data with `make_data_binary()`

RolloTree requires binary (0/1) features. The `make_data_binary()` utility handles one-hot encoding, missing value imputation, and column renaming automatically.

In [5]:
from rollotree.preprocessing.helpers import make_data_binary

# Simulate a raw dataset with mixed types
rng = np.random.RandomState(42)
raw_data = pd.DataFrame({
    "y": rng.choice([1, 2, 3], size=100),
    "age_bin": rng.choice([0, 1], size=100),        # already binary
    "color": rng.choice(["red", "blue", "green"], size=100),  # categorical
    "size": rng.choice(["S", "M", "L", "XL"], size=100),     # categorical
})

# Inject some missing values
raw_data.loc[5, "color"] = None
raw_data.loc[10, "size"] = None

print("Before binarization:")
print(raw_data.head(8))
print(f"\nShape: {raw_data.shape}")
print(f"Dtypes:\n{raw_data.dtypes}")

Before binarization:
   y  age_bin  color size
0  3        1    red    S
1  1        1  green   XL
2  3        1   blue    L
3  3        0    red    L
4  1        0    red    S
5  1        0   None    S
6  3        0  green    L
7  2        1   blue    L

Shape: (100, 4)
Dtypes:
y           int64
age_bin     int64
color      object
size       object
dtype: object


In [6]:
binary_data = make_data_binary(raw_data)

print("After binarization:")
print(binary_data.head(8))
print(f"\nShape: {binary_data.shape}")
print(f"Columns: {list(binary_data.columns)}")
print(f"All values binary: {set(binary_data.drop('y', axis=1).values.flatten()) == {0, 1}}")

After binarization:
   y  1      2      3      4      5      6      7      8
0  3  1  False  False   True  False  False   True  False
1  1  1  False   True  False  False  False  False   True
2  3  1   True  False  False   True  False  False  False
3  3  0  False  False   True   True  False  False  False
4  1  0  False  False   True  False  False   True  False
5  1  0  False  False   True  False  False   True  False
6  3  0  False   True  False   True  False  False  False
7  2  1   True  False  False   True  False  False  False

Shape: (100, 9)
Columns: ['y', 1, 2, 3, 4, 5, 6, 7, 8]
All values binary: True


In [7]:
# Train on the binarized data
X_raw = binary_data.drop("y", axis=1)
y_raw = binary_data["y"]

model_raw = RollingOCT(depth=3, solver="highs")
model_raw.fit(X_raw, y_raw)
print(f"Accuracy on binarized data: {model_raw.score(X_raw, y_raw):.3f}")
print(f"\nTree structure:")
print(export_text(model_raw.tree_))

Accuracy on binarized data: 0.570

Tree structure:
|--- feature_8 == 1
|   |--- feature_4 == 1
|   |   |--- feature_1 == 1
|   |   |   class: 2 {1: 2, 2: 3, 3: 2}
|   |   |--- feature_1 == 0
|   |   |   class: 3 {1: 1, 2: 1, 3: 5}
|   |--- feature_4 == 0
|   |   |--- feature_1 == 1
|   |   |   class: 1 {1: 7, 2: 0, 3: 2}
|   |   |--- feature_1 == 0
|   |   |   class: 2 {1: 5, 2: 7, 3: 1}
|--- feature_8 == 0
|   |--- feature_7 == 1
|   |   |--- feature_4 == 1
|   |   |   class: 1 {1: 6, 2: 1, 3: 2}
|   |   |--- feature_4 == 0
|   |   |   class: 3 {1: 3, 2: 4, 3: 8}
|   |--- feature_7 == 0
|   |   |--- feature_3 == 1
|   |   |   class: 2 {1: 0, 2: 12, 3: 3}
|   |   |--- feature_3 == 0
|   |   |   class: 1 {1: 9, 2: 8, 3: 8}


## 3. Solver Tuning

The MIP solver has several tuning knobs. Here we explore how they affect solution quality and speed.

In [8]:
# Compare solver configurations
configs = [
    {"label": "Default",         "kwargs": {}},
    {"label": "MIP gap 5%",      "kwargs": {"mip_gap": 0.05}},
    {"label": "MIP gap 1%",      "kwargs": {"mip_gap": 0.01}},
    {"label": "Big-M = 10",      "kwargs": {"big_m": 10}},
    {"label": "Big-M = 999",     "kwargs": {"big_m": 999}},
    {"label": "min_leaf=5",      "kwargs": {"min_samples_leaf": 5}},
    {"label": "min_split=20",    "kwargs": {"min_samples_split": 20}},
]

print(f"{'Config':>18} {'Train Acc':>10} {'Test Acc':>10} {'Leaves':>7} {'Time':>8}")
print("-" * 58)

for cfg in configs:
    t0 = time.time()
    m = RollingOCT(depth=3, solver="highs", **cfg["kwargs"])
    m.fit(X_train, y_train)
    elapsed = time.time() - t0
    print(
        f"{cfg['label']:>18} "
        f"{m.score(X_train, y_train):10.4f} "
        f"{m.score(X_test, y_test):10.4f} "
        f"{m.get_n_leaves():7d} "
        f"{elapsed:7.2f}s"
    )

            Config  Train Acc   Test Acc  Leaves     Time
----------------------------------------------------------


           Default     0.6937     0.7778       2    3.03s


        MIP gap 5%     0.6937     0.7778       2    2.93s


        MIP gap 1%     0.6937     0.7778       2    2.99s


        Big-M = 10     0.6937     0.7778       2    2.94s


       Big-M = 999     0.6937     0.7778       2    2.99s


Subproblem at parent 2 failed: SolverStatus.INFEASIBLE — pruning leaves


        min_leaf=5     0.6875     0.8333       4    2.69s


      min_split=20     0.6937     0.7778       2    2.99s


## 4. Parallel Execution

For deeper trees, each depth level may have multiple independent OCT-2 subproblems. `n_jobs` dispatches these across CPU cores.

In [9]:
import os

print(f"Available CPU cores: {os.cpu_count()}")
print()

for depth in [3, 4, 5]:
    # Sequential
    t0 = time.time()
    m_seq = RollingOCT(depth=depth, solver="highs", n_jobs=1)
    m_seq.fit(X_train, y_train)
    t_seq = time.time() - t0

    # Parallel
    t0 = time.time()
    m_par = RollingOCT(depth=depth, solver="highs", n_jobs=-1)
    m_par.fit(X_train, y_train)
    t_par = time.time() - t0

    speedup = t_seq / t_par if t_par > 0 else float("inf")
    print(
        f"Depth {depth}: "
        f"sequential={t_seq:.2f}s, "
        f"parallel={t_par:.2f}s, "
        f"speedup={speedup:.1f}x, "
        f"acc={m_seq.score(X_test, y_test):.3f}"
    )

Available CPU cores: 12



Depth 3: sequential=2.97s, parallel=3.01s, speedup=1.0x, acc=0.778


Depth 4: sequential=4.38s, parallel=4.38s, speedup=1.0x, acc=0.833


Depth 5: sequential=6.44s, parallel=7.17s, speedup=0.9x, acc=0.889


## 5. Tree Internals

Peek under the hood at branch nodes, leaf distributions, and pruning.

In [10]:
# Train a depth-3 model for inspection
model3 = RollingOCT(depth=3, solver="highs")
model3.fit(X_train, y_train)

tree = model3.tree_
print("=== Branch Nodes ===")
for nid in sorted(tree.branch_nodes):
    node = tree.branch_nodes[nid]
    if node.feature_index is not None:
        print(f"  Node {nid}: split on feature {node.feature_index} "
              f"(position {node.feature_position}) "
              f"→ left={node.left_child_id}, right={node.right_child_id}")

=== Branch Nodes ===
  Node 1: split on feature 112 (position 111) → left=2, right=3
  Node 2: split on feature 50 (position 49) → left=4, right=5
  Node 3: split on feature 31 (position 30) → left=6, right=7
  Node 6: split on feature 21 (position 20) → left=12, right=13
  Node 7: split on feature 111 (position 110) → left=14, right=15


In [11]:
print("=== Leaf Nodes ===")
for lid in sorted(tree.leaf_nodes):
    leaf = tree.leaf_nodes[lid]
    if leaf.predicted_class is not None:
        status = "(pruned)" if leaf.is_pruned else "(active)"
        dist = leaf.class_distribution or {}
        total = sum(dist.values()) if dist else 0
        purity = max(dist.values()) / total if total > 0 else 0
        print(f"  Leaf {lid}: class={leaf.predicted_class} {status} "
              f"samples={total} purity={purity:.2f} dist={dist}")

print(f"\nPruned node IDs: {sorted(tree._pruned_node_ids)}")

=== Leaf Nodes ===
  Leaf 4: class=2 (pruned) samples=0 purity=0.00 dist={}
  Leaf 5: class=3 (pruned) samples=0 purity=0.00 dist={}
  Leaf 12: class=2 (pruned) samples=0 purity=0.00 dist={}
  Leaf 13: class=1 (pruned) samples=0 purity=0.00 dist={}
  Leaf 14: class=3 (active) samples=18 purity=1.00 dist={1: 0, 2: 0, 3: 18}
  Leaf 15: class=2 (active) samples=105 purity=0.53 dist={1: 36, 2: 56, 3: 13}

Pruned node IDs: [4, 5, 12, 13]


In [12]:
# Decision path for a single sample — trace the route through the tree
sample_idx = 0
x = X_test.values[sample_idx]
path = model3.decision_path(X_test.values[sample_idx:sample_idx+1])
visited = path.toarray()[0].nonzero()[0]

print(f"Sample {sample_idx} path through tree:")
feature_names = list(X_train.columns)
for nid in visited:
    if nid in tree.branch_nodes:
        node = tree.branch_nodes[nid]
        if node.feature_index is not None:
            feat_name = feature_names[node.feature_position]
            feat_val = x[node.feature_position]
            direction = "left" if feat_val == 1 else "right"
            print(f"  Node {nid}: feature '{feat_name}' = {int(feat_val)} → go {direction}")
    elif nid in tree.leaf_nodes:
        leaf = tree.leaf_nodes[nid]
        print(f"  Leaf {nid}: predict class {leaf.predicted_class}")

print(f"\nActual label: {y_test.values[sample_idx]}")
print(f"Predicted: {model3.predict(X_test.values[sample_idx:sample_idx+1])[0]}")

Sample 0 path through tree:
  Node 1: feature '112' = 0 → go right
  Node 3: feature '31' = 1 → go left
  Node 6: feature '21' = 0 → go right
  Leaf 13: predict class 1

Actual label: 1
Predicted: 1


## 6. Numba Acceleration

When `numba` is installed, tree routing is JIT-compiled for near-C speed. This is especially noticeable on larger datasets.

In [13]:
print(f"Numba available: {HAS_NUMBA}")

# Benchmark prediction speed with a larger synthetic dataset
rng = np.random.RandomState(123)
X_large = rng.randint(0, 2, size=(10000, X_train.shape[1]))

# Warm up (first call triggers JIT compilation if numba is present)
_ = model3.predict(X_large)

n_repeats = 50
t0 = time.time()
for _ in range(n_repeats):
    model3.predict(X_large)
elapsed = (time.time() - t0) / n_repeats

backend = "numba" if HAS_NUMBA else "numpy"
print(f"Backend: {backend}")
print(f"Predict 10,000 samples: {elapsed*1000:.2f} ms")
print(f"Throughput: {10000/elapsed:,.0f} samples/sec")

Numba available: True
Backend: numba
Predict 10,000 samples: 0.20 ms
Throughput: 48,908,603 samples/sec


## 7. Custom Analysis — Per-Leaf Statistics

Use `apply()` to map samples to leaves, then compute per-leaf accuracy, purity, and confusion.

In [14]:
# Per-leaf analysis on test data
leaf_ids = model3.apply(X_test.values)
preds = model3.predict(X_test.values)
y_true = y_test.values

print(f"{'Leaf':>6} {'Samples':>8} {'Correct':>8} {'Accuracy':>9} {'Classes in leaf':>20}")
print("-" * 55)

for lid in sorted(np.unique(leaf_ids)):
    mask = leaf_ids == lid
    n_samples = mask.sum()
    correct = (preds[mask] == y_true[mask]).sum()
    acc = correct / n_samples if n_samples > 0 else 0
    unique_classes = sorted(np.unique(y_true[mask]))
    print(f"{lid:6d} {n_samples:8d} {correct:8d} {acc:9.3f} {str(unique_classes):>20}")

  Leaf  Samples  Correct  Accuracy      Classes in leaf
-------------------------------------------------------
     5        3        3     1.000                  [3]
    12        1        1     1.000                  [2]
    13        3        3     1.000                  [1]
    14        1        1     1.000                  [3]
    15       10        6     0.600            [1, 2, 3]


In [15]:
# Probability calibration check — are predicted probabilities well-calibrated?
proba = model3.predict_proba(X_test.values)
classes = model3.classes_

print("Per-class probability statistics on test set:")
print(f"{'Class':>6} {'Mean P':>8} {'Std P':>8} {'Actual %':>9}")
print("-" * 35)
for i, c in enumerate(classes):
    mean_p = proba[:, i].mean()
    std_p = proba[:, i].std()
    actual_frac = (y_true == c).mean()
    print(f"{c:6d} {mean_p:8.3f} {std_p:8.3f} {actual_frac:9.3f}")

Per-class probability statistics on test set:
 Class   Mean P    Std P  Actual %
-----------------------------------
     1    0.357    0.323     0.333
     2    0.352    0.300     0.389
     3    0.291    0.382     0.278


## 8. Comparing Impurity Criteria Across Depths

RolloTree supports both **Gini index** and **misclassification error** as optimization objectives. Let's compare them systematically.

In [16]:
# Compare Gini vs Misclassification across depths 2-5
results = []
for depth in range(2, 6):
    for criterion in ["gini", "misclassification"]:
        t0 = time.time()
        m = RollingOCT(depth=depth, criterion=criterion, solver="highs")
        m.fit(X_train, y_train)
        elapsed = time.time() - t0
        results.append({
            "depth": depth,
            "criterion": criterion,
            "train_acc": m.score(X_train, y_train),
            "test_acc": m.score(X_test, y_test),
            "n_leaves": m.get_n_leaves(),
            "time": elapsed,
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False, float_format="%.4f"))

 depth         criterion  train_acc  test_acc  n_leaves   time
     2              gini     0.5875    0.6111         4 1.5086
     2 misclassification     0.6188    0.5000         4 1.4392
     3              gini     0.6937    0.7778         2 2.9462
     3 misclassification     0.7063    0.7222         2 2.7664
     4              gini     0.7500    0.8333         3 4.3715
     4 misclassification     0.7875    0.7222         3 4.2352
     5              gini     0.8375    0.8889         4 6.3955
     5 misclassification     0.8688    0.9444         4 6.2222


In [17]:
# Pivot to compare side-by-side
pivot = df_results.pivot(index="depth", columns="criterion", values="test_acc")
print("Test accuracy by depth and criterion:")
print(pivot.to_string(float_format="%.4f"))

print("\nWhich criterion wins at each depth?")
for d in pivot.index:
    gini = pivot.loc[d, "gini"]
    misclass = pivot.loc[d, "misclassification"]
    winner = "gini" if gini >= misclass else "misclassification"
    diff = abs(gini - misclass)
    print(f"  Depth {d}: {winner} wins by {diff:.4f}" if diff > 0 else f"  Depth {d}: tie")

Test accuracy by depth and criterion:
criterion   gini  misclassification
depth                              
2         0.6111             0.5000
3         0.7778             0.7222
4         0.8333             0.7222
5         0.8889             0.9444

Which criterion wins at each depth?
  Depth 2: gini wins by 0.1111
  Depth 3: gini wins by 0.0556
  Depth 4: gini wins by 0.1111
  Depth 5: misclassification wins by 0.0556


In [18]:
# Understand the criteria: manual Gini vs Misclassification calculation
# These are the objectives that the MIP minimizes across leaves.

def manual_gini(class_counts):
    """Weighted Gini impurity for a single leaf."""
    total = sum(class_counts.values())
    if total == 0:
        return 0.0
    sum_sq = sum((c / total) ** 2 for c in class_counts.values())
    return 1.0 - sum_sq

def manual_misclass(class_counts):
    """Misclassification error for a single leaf."""
    total = sum(class_counts.values())
    if total == 0:
        return 0.0
    return 1.0 - max(class_counts.values()) / total

# Compare on example leaf distributions
examples = [
    {"label": "Pure leaf",       "dist": {1: 50, 2: 0}},
    {"label": "Nearly pure",     "dist": {1: 45, 2: 5}},
    {"label": "Moderate split",  "dist": {1: 30, 2: 20}},
    {"label": "Even split",      "dist": {1: 25, 2: 25}},
]

print(f"{'Leaf type':>18} {'Distribution':>16} {'Gini':>8} {'Misclass':>10}")
print("-" * 56)
for ex in examples:
    g = manual_gini(ex["dist"])
    mc = manual_misclass(ex["dist"])
    print(f"{ex['label']:>18} {str(ex['dist']):>16} {g:8.4f} {mc:10.4f}")

print("\nKey insight: Gini penalizes impure leaves more smoothly,")
print("while misclassification only cares about the majority class fraction.")

         Leaf type     Distribution     Gini   Misclass
--------------------------------------------------------
         Pure leaf    {1: 50, 2: 0}   0.0000     0.0000
       Nearly pure    {1: 45, 2: 5}   0.1800     0.1000
    Moderate split   {1: 30, 2: 20}   0.4800     0.4000
        Even split   {1: 25, 2: 25}   0.5000     0.5000

Key insight: Gini penalizes impure leaves more smoothly,
while misclassification only cares about the majority class fraction.


## 9. Feature Importance Deep Dive

Go beyond the basic `feature_importances_` to understand what the tree learned.

In [19]:
# Train a deeper model for richer importances
model5 = RollingOCT(depth=5, solver="highs")
model5.fit(X_train, y_train)

importances = model5.feature_importances_
feature_names = list(X_train.columns)

# Top-10 features
top_k = 10
top_idx = np.argsort(importances)[::-1][:top_k]

print(f"Top {top_k} features (depth-5, {model5.get_n_leaves()} leaves):")
print(f"{'Rank':>4} {'Feature':>10} {'Importance':>11} {'Bar':>20}")
print("-" * 50)
for rank, idx in enumerate(top_idx, 1):
    bar = "█" * int(importances[idx] * 50)
    print(f"{rank:4d} {feature_names[idx]:>10} {importances[idx]:11.4f}  {bar}")

n_used = (importances > 0).sum()
print(f"\nFeatures used: {n_used} / {len(importances)}")
print(f"Unused features: {len(importances) - n_used}")

Top 10 features (depth-5, 4 leaves):
Rank    Feature  Importance                  Bar
--------------------------------------------------
   1         96      0.0909  ████
   2         50      0.0909  ████
   3         99      0.0909  ████
   4         21      0.0909  ████
   5         19      0.0909  ████
   6         31      0.0909  ████
   7        112      0.0909  ████
   8        113      0.0909  ████
   9        111      0.0909  ████
  10         93      0.0909  ████

Features used: 11 / 130
Unused features: 119


In [20]:
# Compare feature usage across depths
print(f"{'Depth':>5} {'Features used':>14} {'Top feature':>12} {'Top importance':>15}")
print("-" * 50)

for depth in range(2, 6):
    m = RollingOCT(depth=depth, solver="highs")
    m.fit(X_train, y_train)
    imp = m.feature_importances_
    n_used = (imp > 0).sum()
    top_feat = feature_names[imp.argmax()]
    top_val = imp.max()
    print(f"{depth:5d} {n_used:14d} {top_feat:>12} {top_val:15.4f}")

Depth  Features used  Top feature  Top importance
--------------------------------------------------


    2              3           50          0.3333


    3              5           21          0.2000


    4              7           21          0.1429


    5             11           19          0.0909


## 10. Early Stopping via `min_samples_split` and `min_samples_leaf`

These parameters control tree complexity by preventing splits on small subsets.

In [21]:
# Effect of min_samples_split on tree complexity
print("min_samples_split effect (depth=5):")
print(f"{'min_split':>10} {'Train Acc':>10} {'Test Acc':>10} {'Leaves':>7} {'Depth':>6}")
print("-" * 48)

for min_split in [2, 5, 10, 20, 40]:
    m = RollingOCT(depth=5, solver="highs", min_samples_split=min_split)
    m.fit(X_train, y_train)
    print(
        f"{min_split:10d} "
        f"{m.score(X_train, y_train):10.4f} "
        f"{m.score(X_test, y_test):10.4f} "
        f"{m.get_n_leaves():7d} "
        f"{m.get_depth():6d}"
    )

min_samples_split effect (depth=5):
 min_split  Train Acc   Test Acc  Leaves  Depth
------------------------------------------------


         2     0.8375     0.8889       4      5


         5     0.8375     0.8889       4      5


        10     0.8375     0.8889       4      5


        20     0.8313     0.8889       4      5


        40     0.8313     0.8889       4      5


In [22]:
# Effect of min_samples_leaf
print("\nmin_samples_leaf effect (depth=5):")
print(f"{'min_leaf':>10} {'Train Acc':>10} {'Test Acc':>10} {'Leaves':>7} {'Depth':>6}")
print("-" * 48)

for min_leaf in [1, 3, 5, 10, 20]:
    m = RollingOCT(depth=5, solver="highs", min_samples_leaf=min_leaf)
    m.fit(X_train, y_train)
    print(
        f"{min_leaf:10d} "
        f"{m.score(X_train, y_train):10.4f} "
        f"{m.score(X_test, y_test):10.4f} "
        f"{m.get_n_leaves():7d} "
        f"{m.get_depth():6d}"
    )


min_samples_leaf effect (depth=5):
  min_leaf  Train Acc   Test Acc  Leaves  Depth
------------------------------------------------


         1     0.8375     0.8889       4      5


Subproblem at parent 13 failed: SolverStatus.INFEASIBLE — pruning leaves


         3     0.8313     0.9444       6      5


Subproblem at parent 2 failed: SolverStatus.INFEASIBLE — pruning leaves


         5     0.8313     0.8889       4      5


        10     0.8375     0.8333       4      5


        20     0.7937     0.7778       2      5


## Summary

| Feature | Code |
|---------|------|
| Per-depth metrics | `model.depth_results_[d].training_accuracy` |
| Binarize raw data | `make_data_binary(df)` |
| MIP gap tuning | `RollingOCT(mip_gap=0.01)` |
| Parallel solving | `RollingOCT(n_jobs=-1)` |
| Branch node inspection | `tree.branch_nodes[nid].feature_index` |
| Leaf distributions | `tree.leaf_nodes[lid].class_distribution` |
| Pruned nodes | `tree._pruned_node_ids` |
| Numba check | `from rollotree.tree._numba import HAS_NUMBA` |
| Per-leaf analysis | `model.apply(X)` to get leaf IDs |
| Impurity criteria | `GiniCriterion()`, `MisclassificationCriterion()` |
| Early stopping | `min_samples_split`, `min_samples_leaf` |

See **01_quickstart.ipynb** for basics, **02_visualization.ipynb** for tree plots, **03_sklearn_integration.ipynb** for GridSearchCV and Pipeline.